[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/OL7014-integer-programming/blob/main/notebooks/week1_knapsack.ipynb)

In [ ]:
%pip install -q gurobipy

# Knapsack — an explicit 0/1 integer program

Same instance as `ip-games/knapsack.html`: a resupply rucksack, 6 candidate items, one weight
capacity. The math, with the actual numbers this notebook uses:

$$\max \;\; 9x_{\text{rifle}} + 8x_{\text{boots}} + 9x_{\text{helmet}} + 8x_{\text{canteen}} + 3x_{\text{medkit}} + 9x_{\text{rations}}$$

$$\text{s.t.} \;\; 5x_{\text{rifle}} + 7x_{\text{boots}} + 8x_{\text{helmet}} + 8x_{\text{canteen}} + 8x_{\text{medkit}} + 2x_{\text{rations}} \le 21$$

$$x_j \in \{0, 1\} \quad \text{for every item } j$$

Each coefficient is (value in pts) in the objective and (weight in kg) in the constraint — the
same six numbers, reused exactly, appear in the code below.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
m = gp.Model("Knapsack")

# decision variables, we have six binary variables --- e.g., m.addVar(vtype=GRB.BINARY), m.addVar(vtype=GRB.CONTINUOUS)


# let's define objective function --- m.setObjective()


# now constraints --- m.addConstr()


# Model is set up. Let's optimize --- m.optimize()


# Let's print the optimal solution

## Reference: the completed model

For after class (or if you get stuck live) — everything typed out by hand, no dictionaries, no loops, no `gp.quicksum`, so every term of the objective and the constraint matches the math above exactly.

In [ ]:
m = gp.Model("knapsack")

# decision variables -- one binary per item
x_rifle   = m.addVar(vtype=GRB.BINARY, name="x_rifle")
x_boots   = m.addVar(vtype=GRB.BINARY, name="x_boots")
x_helmet  = m.addVar(vtype=GRB.BINARY, name="x_helmet")
x_canteen = m.addVar(vtype=GRB.BINARY, name="x_canteen")
x_medkit  = m.addVar(vtype=GRB.BINARY, name="x_medkit")
x_rations = m.addVar(vtype=GRB.BINARY, name="x_rations")

# objective -- total value (pts), written out term by term
m.setObjective(
    9 * x_rifle + 8 * x_boots + 9 * x_helmet + 8 * x_canteen + 3 * x_medkit + 9 * x_rations,
    GRB.MAXIMIZE,
)

# constraint -- total weight (kg) within the 21 kg capacity, written out term by term
m.addConstr(
    5 * x_rifle + 7 * x_boots + 8 * x_helmet + 8 * x_canteen + 8 * x_medkit + 2 * x_rations <= 21,
    name="capacity",
)

m.optimize()

print(f"\nrifle={x_rifle.X:.0f}  boots={x_boots.X:.0f}  helmet={x_helmet.X:.0f}  "
      f"canteen={x_canteen.X:.0f}  medkit={x_medkit.X:.0f}  rations={x_rations.X:.0f}")
print(f"value = {m.ObjVal:.0f} pts")

## Plot the solution

One bar per item, colored differently, height = $x_j$ (1 = packed, 0 = left behind). The title
carries the optimal objective value. We'll reuse this exact plot later to compare against the
**LP relaxation** — there, bars can land anywhere between 0 and 1.

In [ ]:
import matplotlib.pyplot as plt

items = ["rifle", "boots", "helmet", "canteen", "medkit", "rations"]
x_vals = [x_rifle.X, x_boots.X, x_helmet.X, x_canteen.X, x_medkit.X, x_rations.X]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#CCB974"]

plt.figure(figsize=(6, 4))
plt.bar(items, x_vals, color=colors)
plt.ylim(0, 1.1)
plt.ylabel("packed? ($x_j$)")
plt.title(f"Optimal packing — value = {m.ObjVal:.0f} pts")
plt.tight_layout()
plt.show()

## Fancier version (same model) — dictionary + `gp.quicksum`

The version above doesn't scale: 20 items means 20 lines of variables and two 20-term sums typed by hand. This equivalent version loops over a dictionary instead, so it works for any number of items without rewriting the model. Left commented out for now — same answer, less typing, less obvious which term is which.

In [ ]:
# items = {
#     "rifle":   (5, 9),   # (weight kg, value pts)
#     "boots":   (7, 8),
#     "helmet":  (8, 9),
#     "canteen": (8, 8),
#     "medkit":  (8, 3),
#     "rations": (2, 9),
# }
# CAPACITY = 21
#
# m2 = gp.Model("knapsack_dict")
# x = m2.addVars(items.keys(), vtype=GRB.BINARY, name="x")
# m2.setObjective(gp.quicksum(items[j][1] * x[j] for j in items), GRB.MAXIMIZE)
# m2.addConstr(gp.quicksum(items[j][0] * x[j] for j in items) <= CAPACITY, name="capacity")
# m2.optimize()
# print([j for j in items if x[j].X > 0.5], m2.ObjVal)